<a href="https://colab.research.google.com/github/Zong0120/Where_is_Waldo/blob/YOLOv8/Yolov8_model_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
  import ultralytics
  print("Ultralytics 已安裝，版本:", ultralytics.__version__)
except ImportError:
  print("Ultralytics 未安裝，正在安裝...")
  !pip install ultralytics
  from ultralytics import YOLO
  print("Ultralytics 安裝完成，版本:", ultralytics.__version__)
import torch
import cv2
import matplotlib.pyplot as plt
import os

In [ ]:
#取得資料夾所有符合附檔名的檔案
def get_files_path(folder_path, file_extension):

  all_files = os.listdir(folder_path)

  files = [f for f in all_files if any(f.endswith(ext) for ext in file_extension)]

  files_path = [os.path.join(folder_path, single_file) for single_file in files]

  return files_path

In [ ]:
#預測模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = os.path.join(drive_save_path, "saved_models")

best_model_path = os.path.join(model_path, "best.pt")
source_path = os.path.join(colab_save_path, folder_name_split, "test", "images")
result_path = os.path.join(colab_save_path, "runs", "detect", "predict")

model = YOLO(best_model_path).to(device)
result_test = model.predict(source_path, conf=0.7, save=True, save_txt=True, save_conf=True, device=device)

In [ ]:
def convert_label_to_location(txt_path, block_size):

  with open(txt_path, "r") as f:
    labels = [line.strip() for line in f]

  label_list = []

  for label_str in labels:
    label = label_str.split(" ")

    #還原座標
    label_id = int(label[0])
    label_xy = (int(float(label[1]) * block_size), int(float(label[2]) * block_size))
    label_wh = (int(float(label[3]) * block_size), int(float(label[4]) * block_size))
    label_confidence = float(label[5])

    label_list.append((label_id, label_xy, label_wh, label_confidence))

  return label_list

def compute_iou(box1, box2):

  cx1, cy1, w1, h1 = box1
  cx2, cy2, w2, h2 = box2

  #轉換為(x_min, y_min, x_max, y_max)
  x1_min, y1_min = cx1 - w1 / 2, cy1 - h1 / 2
  x1_max, y1_max = cx1 + w1 / 2, cy1 + h1 / 2

  x2_min, y2_min = cx2 - w2 / 2, cy2 - h2 / 2
  x2_max, y2_max = cx2 + w2 / 2, cy2 + h2 / 2

  #交集
  inter_x_min = max(x1_min, x2_min)
  inter_y_min = max(y1_min, y2_min)
  inter_x_max = min(x1_max, x2_max)
  inter_y_max = min(y1_max, y2_max)

  inter_width = max(0, inter_x_max - inter_x_min)
  inter_height = max(0, inter_y_max - inter_y_min)
  intersection_area = inter_width * inter_height

  #聯集
  area1 = w1 * h1
  area2 = w2 * h2
  union_area = area1 + area2 - intersection_area

  iou = intersection_area / union_area if union_area > 0 else 0.0

  return iou

#刪除重疊的預測框
def non_maximum_suppression(boxes, iou_threshold=0.5):

  #根據類別分組，避免不同類別互相影響
  grouped_boxes = {}
  for box in boxes:
    class_id = box[0]
    if class_id not in grouped_boxes:
      grouped_boxes[class_id] = []
    grouped_boxes[class_id].append(box)

  #存放最終的NMS結果
  final_boxes = []

  for class_id, class_boxes in grouped_boxes.items():
    #信心度從高排到低
    class_boxes.sort(key=lambda b: b[5], reverse=True)

    #篩選過的預測框
    selected_boxes = []

    while class_boxes:
      #取出信心度最高的框
      best_box = class_boxes.pop(0)
      selected_boxes.append(best_box)

      #過濾IoU過高的框
      class_boxes = [
        box for box in class_boxes
        if compute_iou(best_box[1:5], box[1:5]) < iou_threshold
      ]

    #加入結果
    final_boxes.extend(selected_boxes)

  return final_boxes

In [ ]:
#顯示預測結果
import cv2
def process_images(image_folder, txt_folder, output_folder, block_size, stride):

    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    #取得原圖路徑、txt路徑
    image_paths = get_files_path(image_folder, ".jpg")
    txt_paths = get_files_path(txt_folder, ".txt")

    #建立txt檔案的字典
    txt_dict = {}
    for txt_path in txt_paths:
      txt_name = os.path.basename(txt_path)
      base_name = txt_name.split('_')[0]
      if base_name not in txt_dict:
        txt_dict[base_name] = []
      txt_dict[base_name].append(txt_path)

    #儲存原圖上的預測框
    all_boxes = {}

    for image_path in image_paths:
      image = cv2.imread(image_path)
      img_height, img_width = image.shape[:2]

      #原圖的檔名
      base_name = image_path.split("/")[-1].split(".")[0]

      #如果沒有對應的txt檔則跳過
      if base_name not in txt_dict:
        print(f"{base_name}.jpg 沒有txt檔案")
        continue

      for txt_path in txt_dict[base_name]:
        txt_name = os.path.basename(txt_path)
        parts = os.path.splitext(txt_name)[0].split('_')

        if len(parts) < 3:
          print(f"{txt_name}.txt 檔名格式錯誤")

        try:
          row_index, col_index = int(parts[1].lstrip("0") or "0"), int(parts[2].lstrip("0") or "0")
        except ValueError:
          print(f"{txt_name}.txt 無法拆出row_index和col_index")
          continue

        #區塊在原圖的座標
        if row_index * stride + block_size > img_height:
          block_y = img_height - block_size
        else:
          block_y = row_index * stride

        if col_index * stride + block_size > img_width:
          block_x = img_width - block_size
        else:
          block_x = col_index * stride

        boxes = convert_label_to_location(txt_path, block_size)

        for id, (x, y), (w, h), conf in boxes:
          #預測框在原圖的座標
          orig_x, orig_y = block_x + x, block_y + y

          if base_name not in all_boxes:
              all_boxes[base_name] = []
          all_boxes[base_name].append((id, orig_x, orig_y, w, h, conf))

    #合併靠近的物件框
    for image_name, boxes in all_boxes.items():
      merged_boxes = non_maximum_suppression(boxes)

      image_path = os.path.join(image_folder, image_name + ".jpg")
      image = cv2.imread(image_path).copy()
      #所有預測類別
      class_names = model.names

      for id, x, y, w, h, conf in merged_boxes:

        x1 = x - w // 2
        y1 = y - h // 2
        x2 = x + w // 2
        y2 = y + h // 2

        #繪製預測框
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), 3)
        #加上文字
        text = f"{class_names[id]} {conf:.2}"
        text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)[0]
        text_width, text_height = text_size
        cv2.rectangle(image, (x1, y1 - text_height - 20), (x1 + text_width, y1), (0, 0, 255), -1)
        cv2.putText(image, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)

      output_path = os.path.join(output_folder, image_name + "_out.jpg")
      cv2.imwrite(output_path, image)

#測試函式
process_images(
    image_folder=os.path.join(colab_save_path, folder_name, folder_list[2], folder_list_2[0]),
    txt_folder="runs/detect/predict/labels",
    output_folder=os.path.join(colab_save_path, "output"),
    block_size=640,
    stride=160
)
image_paths = get_files_path(os.path.join(colab_save_path, "output"), ".jpg")

for image_path in image_paths:
  img = Image.open(image_path)
  plt.figure(figsize=(10, 10))
  plt.imshow(img)
  plt.show()

#儲存到Google Drive
shutil.copytree(os.path.join(colab_save_path, "output"), os.path.join(drive_save_path, "output"))